# Bradford Bulls — Team Detection & Classification

## Approach: SigLIP embeddings + K-means (SOTA 2025)

Uses **SigLIP** (Google vision-language model) to extract rich 768-dim appearance embeddings
from each player crop — captures jersey design, colour, logo, texture holistically.
Far more robust than raw colour histograms, especially when teams wear dark/mixed kits.

**Pipeline:**
1. YOLO26n + ByteTrack detects and tracks every player
2. SigLIP extracts appearance embedding per crop (grass-removed upper body)
3. K-means(k=3) on warm-up embeddings → 3 clusters
4. You assign cluster 0/1/2 → Bradford / Opponent / Referee
5. Majority-vote temporal smoothing per track ID → stable labels

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'Device  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — switch to T4"}')

## 2. Install

In [ ]:
!pip install -q ultralytics transformers
print('Done')

## 3. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# >>> UPDATE path to your video <<<
VIDEO_PATH = '/content/drive/MyDrive/bradford_bulls/videos/M02_h264.mp4'

# ── Detection ─────────────────────────────────────────────────────────────────
CONF_THRESH       = 0.50   # YOLO person confidence
MIN_PLAYER_HEIGHT = 0.07   # min bbox height / frame height (filters spectators)

# ── Warm-up ───────────────────────────────────────────────────────────────────
WARMUP_FRAMES     = 40     # frames sampled for K-means fitting

# ── Temporal smoothing ────────────────────────────────────────────────────────
SMOOTHING_WINDOW  = 20     # majority vote over last N frames per track ID

# ── SigLIP inference ──────────────────────────────────────────────────────────
SIGLIP_BATCH      = 8      # player crops processed together (speed vs memory)

# ── Bounding box colours per team (BGR) ───────────────────────────────────────
BOX_COLORS = {
    'Team A':  (255, 80,  80 ),   # blue  — Bradford
    'Team B':  (80,  80,  255),   # red   — Opponent
    'Referee': (0,   220, 220),   # yellow
}

# ── Output ────────────────────────────────────────────────────────────────────
stem       = Path(VIDEO_PATH).stem
OUTPUT_DIR = Path(f'/content/drive/MyDrive/bradford_bulls/team_detection/{stem}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Video  : {VIDEO_PATH}')
print(f'Exists : {Path(VIDEO_PATH).exists()}')
print(f'Output : {OUTPUT_DIR}')

## 4. Load Models — YOLO26n + SigLIP

In [ ]:
import torch
from ultralytics import YOLO
from transformers import AutoProcessor, AutoModel
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── YOLO26n ────────────────────────────────────────────────────────────────────
yolo = YOLO('yolo26n.pt')
print('YOLO26n ready')

# ── SigLIP ─────────────────────────────────────────────────────────────────────
SIGLIP_NAME  = 'google/siglip-base-patch16-224'
siglip_proc  = AutoProcessor.from_pretrained(SIGLIP_NAME)
siglip_model = AutoModel.from_pretrained(SIGLIP_NAME).to(DEVICE).eval()
print(f'SigLIP ready on {DEVICE.upper()}')


@torch.no_grad()
def get_embeddings(crops_rgb):
    """
    crops_rgb : list of H×W×3 uint8 numpy arrays
    Returns   : (N, D) float32 numpy, L2-normalised
    """
    pil_list       = [Image.fromarray(c) for c in crops_rgb]
    inputs         = siglip_proc(images=pil_list, return_tensors='pt',
                                 padding=True).to(DEVICE)
    vision_outputs = siglip_model.vision_model(**inputs)
    feats          = vision_outputs.pooler_output          # (N, hidden_dim)
    feats          = feats / feats.norm(dim=-1, keepdim=True)   # L2 normalise
    return feats.cpu().numpy()

## 5. Helper Functions

In [ ]:
import cv2
import numpy as np


def crop_upper_body(frame_rgb, box):
    """
    Return the upper-body crop (top 55% of bbox, 10% inset on sides).
    This captures the jersey and avoids shorts + grass background.
    Returns None if crop is empty.
    """
    x1, y1, x2, y2 = [int(v) for v in box]
    h, w = y2 - y1, x2 - x1
    cy2  = y1 + int(0.55 * h)
    cx1  = x1 + int(0.10 * w)
    cx2  = x2 - int(0.10 * w)
    if cy2 <= y1 or cx2 <= cx1:
        return None
    crop = frame_rgb[y1:cy2, cx1:cx2]
    return crop if crop.size > 0 else None


def draw_label_box(img_bgr, box, label, color_bgr):
    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color_bgr, 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
    cv2.rectangle(img_bgr, (x1, y1 - th - 6), (x1 + tw + 6, y1), color_bgr, -1)
    cv2.putText(img_bgr, label, (x1 + 3, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)


def read_frame_safe(video_path, target_sec):
    """Read a frame at target_sec; falls back to sequential read if seek fails."""
    cap   = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    dur   = total / fps
    t     = max(0.0, min(target_sec, dur * 0.85))
    fidx  = int(t * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
    ok, frame = cap.read()
    if not ok:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        last = None
        for i in range(total):
            ok2, f = cap.read()
            if ok2:
                last = f
                if i >= fidx:
                    break
        frame = last
    cap.release()
    return frame, fps, total, dur


print('Helper functions ready')

## 6. Warm-up — SigLIP Embeddings + K-means Clustering

Samples frames evenly across the video, extracts SigLIP embeddings for every player,
then clusters into 3 groups.  Shows representative crops per cluster so you can
assign team labels in the next cell.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm

# ── Video info ────────────────────────────────────────────────────────────────
cap     = cv2.VideoCapture(VIDEO_PATH)
fps_v   = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
H_frame = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
duration = total_f / fps_v
MIN_H_PX = int(MIN_PLAYER_HEIGHT * H_frame)
print(f'Video: {duration:.1f}s  ({total_f} frames @ {fps_v:.0f} fps)')

warmup_idx = np.linspace(0, total_f - 1, WARMUP_FRAMES, dtype=int)

all_embeddings = []   # (N, D)
all_crops      = []   # numpy crops for display

cap = cv2.VideoCapture(VIDEO_PATH)
for fidx in tqdm(warmup_idx, desc='Warm-up SigLIP'):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
    ok, frame = cap.read()
    if not ok:
        continue
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = yolo(frame, verbose=False, conf=CONF_THRESH, classes=[0])
    crops_batch, boxes_batch = [], []
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        if (y2 - y1) < MIN_H_PX:
            continue
        crop = crop_upper_body(frame_rgb, (x1, y1, x2, y2))
        if crop is not None:
            crops_batch.append(crop)
            boxes_batch.append((x1, y1, x2, y2))

    if not crops_batch:
        continue

    for i in range(0, len(crops_batch), SIGLIP_BATCH):
        batch = crops_batch[i:i + SIGLIP_BATCH]
        embs  = get_embeddings(batch)
        all_embeddings.extend(embs)
        all_crops.extend(batch)
cap.release()

print(f'Collected {len(all_embeddings)} player embeddings')

# ── K-means(k=3) in SigLIP embedding space ────────────────────────────────────
emb_arr = np.array(all_embeddings)   # (N, D)
kmeans  = KMeans(n_clusters=3, random_state=42, n_init=15)
kmeans.fit(emb_arr)
labels  = kmeans.labels_

# ── Show 5 representative crops per cluster ───────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(12, 8))
for cid in range(3):
    idxs = np.where(labels == cid)[0]
    # Pick crops closest to centroid
    centroid = kmeans.cluster_centers_[cid]
    dists    = np.linalg.norm(emb_arr[idxs] - centroid, axis=1)
    best_5   = idxs[np.argsort(dists)[:5]]
    for j, idx in enumerate(best_5):
        axes[cid][j].imshow(all_crops[idx])
        axes[cid][j].axis('off')
    for j in range(len(best_5), 5):
        axes[cid][j].axis('off')
    axes[cid][0].set_ylabel(f'Cluster {cid}\n({len(idxs)} players)',
                             fontsize=10, fontweight='bold')

plt.suptitle('K-means clusters (SigLIP embeddings)\n'
             '→ Identify each cluster in the next cell',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kmeans_clusters.png', dpi=120, bbox_inches='tight')
plt.show()
print('→ Update TEAM_A_CLUSTER / TEAM_B_CLUSTER / REFEREE_CLUSTER in the next cell.')

## 7. Assign Cluster Labels  ← **UPDATE HERE**

Look at the 3×5 grid above — each row is one cluster.
Set which cluster number corresponds to Bradford / Opponent / Referee.

In [ ]:
# >>> UPDATE based on the cluster grid above <<<
TEAM_A_CLUSTER   = 0   # Bradford Bulls
TEAM_B_CLUSTER   = 1   # Opponent
REFEREE_CLUSTER  = 2   # Referee (set same as one of the above if no referee visible)

TEAM_A_LABEL = 'Bradford'
TEAM_B_LABEL = 'Opponent'
REF_LABEL    = 'Referee'

CLUSTER_MAP = {
    TEAM_A_CLUSTER:  'Team A',
    TEAM_B_CLUSTER:  'Team B',
    REFEREE_CLUSTER: 'Referee',
}
LABEL_MAP = {'Team A': TEAM_A_LABEL, 'Team B': TEAM_B_LABEL, 'Referee': REF_LABEL}

# Pre-compute per-team embedding centroids from K-means
TEAM_CENTROIDS = {}
for cid, team in CLUSTER_MAP.items():
    TEAM_CENTROIDS[team] = kmeans.cluster_centers_[cid]
    count = (kmeans.labels_ == cid).sum()
    print(f'Cluster {cid} → {LABEL_MAP[team]:12s}  ({count} warm-up samples)')


def classify_embedding(emb):
    """Cosine similarity to pre-computed team centroids."""
    best_team, best_sim = 'Team A', -1.0
    for team, centroid in TEAM_CENTROIDS.items():
        sim = float(np.dot(emb, centroid))
        if sim > best_sim:
            best_sim, best_team = sim, team
    return best_team

## 8. Quick Verify — Classify Warm-up Crops

In [ ]:
# Classify all warm-up crops and show 5 samples per assigned team
team_crops = {'Team A': [], 'Team B': [], 'Referee': []}
for emb, crop in zip(all_embeddings, all_crops):
    t = classify_embedding(np.array(emb))
    team_crops[t].append(crop)

fig, axes = plt.subplots(3, 5, figsize=(12, 8))
color_map = {'Team A': '#5050FF', 'Team B': '#FF5050', 'Referee': '#00CCCC'}
for row, (team, crops) in enumerate(team_crops.items()):
    for col in range(5):
        ax = axes[row][col]
        if col < len(crops):
            ax.imshow(crops[col])
        ax.axis('off')
    axes[row][0].set_ylabel(
        f'{LABEL_MAP[team]}\n({len(crops)} total)',
        fontsize=10, color=color_map[team], fontweight='bold'
    )

plt.suptitle('Classification verify — if wrong, update cluster numbers in cell above',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'classify_verify.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Generate Team Detection Video

In [ ]:
import subprocess
from collections import deque, Counter
from tqdm.notebook import tqdm

cap      = cv2.VideoCapture(VIDEO_PATH)
src_fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_vid    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
MIN_H_PX = int(MIN_PLAYER_HEIGHT * H_vid)

TMP_PATH      = '/tmp/team_detect_raw.mp4'
OUT_VIDEO     = OUTPUT_DIR / f'{stem}_team_detection.mp4'
writer        = cv2.VideoWriter(TMP_PATH, cv2.VideoWriter_fourcc(*'mp4v'), src_fps, (W, H_vid))
track_history = {}   # track_id → deque of raw labels
timeline      = []

for fidx in tqdm(range(total_f), desc='Team detection'):
    ok, frame = cap.read()
    if not ok:
        break

    output    = frame.copy()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    counts    = {'Team A': 0, 'Team B': 0, 'Referee': 0}

    # ByteTrack for stable track IDs
    results = yolo.track(frame, persist=True, verbose=False,
                         conf=CONF_THRESH, classes=[0],
                         tracker='bytetrack.yaml')

    if results[0].boxes.id is not None:
        # Batch all valid crops for this frame → single SigLIP forward pass
        valid = []
        for box, tid in zip(results[0].boxes,
                            results[0].boxes.id.int().tolist()):
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            if (y2 - y1) < MIN_H_PX:
                continue
            crop = crop_upper_body(frame_rgb, (x1, y1, x2, y2))
            if crop is not None:
                valid.append((x1, y1, x2, y2, tid, crop))

        if valid:
            crops_batch = [v[5] for v in valid]
            embs = get_embeddings(crops_batch)   # (N, D) — one forward pass

            for (x1, y1, x2, y2, tid, _), emb in zip(valid, embs):
                raw_team = classify_embedding(emb)

                # Temporal smoothing: majority vote
                if tid not in track_history:
                    track_history[tid] = deque(maxlen=SMOOTHING_WINDOW)
                track_history[tid].append(raw_team)
                stable_team = Counter(track_history[tid]).most_common(1)[0][0]

                label     = LABEL_MAP.get(stable_team, stable_team)
                color_bgr = BOX_COLORS.get(stable_team, (180, 180, 180))
                draw_label_box(output, (x1, y1, x2, y2), label, color_bgr)
                if stable_team in counts:
                    counts[stable_team] += 1

    writer.write(output)
    timeline.append({'frame': fidx, 'sec': round(fidx / src_fps, 2),
                     'Team A': counts['Team A'], 'Team B': counts['Team B'],
                     'Referee': counts['Referee']})

cap.release()
writer.release()

subprocess.run(
    ['ffmpeg', '-y', '-i', TMP_PATH,
     '-c:v', 'libx264', '-preset', 'fast', '-crf', '20', str(OUT_VIDEO)],
    check=True, capture_output=True
)
print(f'Video → {OUT_VIDEO}  ({OUT_VIDEO.stat().st_size/1e6:.1f} MB)')
print(f'Track IDs seen: {len(track_history)}')

## 10. Timeline Chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(timeline)
w  = max(1, int(src_fps))
for col in ['Team A', 'Team B', 'Referee']:
    df[f'{col}_s'] = df[col].rolling(w, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df['sec'], df['Team A_s'],  color='#5050FF', label=TEAM_A_LABEL, lw=1.5)
axes[0].plot(df['sec'], df['Team B_s'],  color='#FF5050', label=TEAM_B_LABEL, lw=1.5)
axes[0].plot(df['sec'], df['Referee_s'], color='#00CCCC', label=REF_LABEL,    lw=1.0, ls='--')
axes[0].set_ylabel('Players / frame'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

total = df['Team A_s'] + df['Team B_s'] + 1e-9
axes[1].stackplot(df['sec'],
                  df['Team A_s'] / total * 100,
                  df['Team B_s'] / total * 100,
                  colors=['#5050FF', '#FF5050'], alpha=0.6,
                  labels=[f'{TEAM_A_LABEL} %', f'{TEAM_B_LABEL} %'])
axes[1].set_ylabel('Relative presence (%)'); axes[1].set_xlabel('Time (s)')
axes[1].set_ylim(0, 100); axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle(f'Team Presence — {stem}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'team_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Export CSV

In [ ]:
csv_path = OUTPUT_DIR / f'{stem}_team_timeline.csv'
df[['frame', 'sec', 'Team A', 'Team B', 'Referee']].to_csv(csv_path, index=False)
print(f'CSV → {csv_path}\n')
print('── Summary ──────────────────────────────────────────')
for team, label in LABEL_MAP.items():
    col = team
    print(f'{label:12s}  avg {df[col].mean():.1f}/frame   max {int(df[col].max())}/frame')